In [1]:
!python3 --version

Python 3.13.3


In [2]:
!pip install crewai crewai_tools

  Using cached appdirs-1.4.4-py2.py3-none-any.whl.metadata (9.0 kB)
  Using cached chromadb-1.1.1-cp39-abi3-macosx_11_0_arm64.whl.metadata (7.2 kB)
  Using cached jsonref-1.1.0-py3-none-any.whl.metadata (2.7 kB)
  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached portalocker-2.7.0-py2.py3-none-any.whl.metadata (6.8 kB)
  Using cached pydantic_settings-2.10.1-py3-none-any.whl.metadata (3.4 kB)
  Using cached python_dotenv-1.1.1-py3-none-any.whl.metadata (24 kB)
  Using cached docker-7.1.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached python_docx-1.2.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached pytube-15.0.0-py3-none-any.whl.metadata (5.0 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached youtube_transcript_api-1.2.3-py3-none-any.whl.metadata (24 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached numpy-2.4.1-cp313-cp313-macosx_14_0_arm64.whl.metadata (6.6 kB)
  Using ca

In [3]:
from dotenv import load_dotenv

load_dotenv()  # 현재 경로의 .env 파일을 불러와 환경변수로 등록

True

In [ ]:
# 에이전트 

In [4]:
from crewai import Agent

# 블로그 콘텐츠 생성 에이전트
blog_agent = Agent(
    role="블로그 콘텐츠 생성 에이전트",
    goal=(
        "주어진 주제에 대해 짧고 임팩트 있는 블로그 제목과, "
        "200단어 이내의 본문을 작성하는 것이 목표입니다. "
        "독자가 핵심 개념과 메시지를 빠르게 이해하도록 돕습니다."
    ),
    backstory=(
        "당신은 IT·AI·데이터 분야에서 10년 이상 활동해 온 시니어 콘텐츠 마케터입니다. "
        "기술 블로그, 뉴스레터, 발표 자료 등 다양한 포맷을 제작해 왔고, "
        "복잡한 기술 개념을 비전공자도 이해할 수 있는 언어로 풀어 쓰는 능력이 뛰어납니다. "
        "항상 '독자가 이 글에서 무엇을 얻어 가는가?'를 기준으로 내용을 구성합니다."
    ),
    verbose=False,
    llm="gpt-4o"
)


In [ ]:
# 작업 

In [12]:
from crewai import Agent, Task
from pydantic import BaseModel

# 블로그 출력 구조 정의(Pydantic) → Task 결과 형식 강제
class Blog(BaseModel):
    title: str
    content: str

# 1) 블로그 글 생성 Task: 원본 콘텐츠 만들기
write_blog_task = Task(
    description=(
        "{topic}에 대한 블로그 제목과 본문을 작성하라. "
        "본문은 200단어 이내로 구성하고, 비전공자도 이해할 수 있게 설명한다."
    ),
    expected_output=(
        "다음 필드를 갖는 결과를 생성한다.\n"
        "- title: 블로그 제목 한 문장\n"
        "- content: 200단어 이내의 블로그 본문"
    ),
    agent=blog_agent,          # 블로그 작성 전담 에이전트
    output_pydantic=Blog       # 출력 형식을 Blog 모델로 강제
)

# 2) 생성된 글을 기반으로 요약·정리 Task: 2차 가공 + 파일 저장
save_task = Task(
    description=(
        "이전 Task에서 생성한 블로그 글을 바탕으로, "
        "핵심 내용을 요약하고 주요 포인트를 정리하라. "
        "경영진이 빠르게 읽을 수 있는 요약본 형식으로 작성한다."
    ),
    expected_output=(
        "다음 구조의 Markdown 텍스트를 생성한다.\n"
        "1. 3문장 이내 전체 요약\n"
        "2. 핵심 포인트 3~5개를 불릿 리스트로 정리"
    ),
    agent=blog_agent,
    context=[write_blog_task],       # 이전 Task 결과를 입력으로 사용
    markdown=True,                   # Markdown 포맷으로 결과 생성
    output_file="output/blog_summary.md",  # 요약본을 파일로 저장
    create_directory=True            # 경로가 없으면 자동 생성
)


In [ ]:
from crewai import Crew, Process

crew = Crew(
    agents=[blog_agent],
    tasks=[write_blog_task, save_task],
    process=Process.sequential,
    verbose=True
)

result = crew.kickoff(inputs={"topic": "에이전트"})

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  ea1ebf5a-10ef-4393-84f1-5cfe0a647043                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 에이전트에 대한 블로그 제목과 본문을 작성하라. 본문은 200단어 이내로 구성하고, 비전공자도 이해할 수      │
│  있게 설명한다.                                                                                                 │
│  ID: 877c607c-1d29-4c57-be9d-3a13f3a4a878                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 블로그 콘텐츠 생성 에이전트                                                                             │
│                                                                                                                 │
│  Task: 에이전트에 대한 블로그 제목과 본문을 작성하라. 본문은 200단어 이내로 구성하고, 비전공자도 이해할 수      │
│  있게 설명한다.                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 블로그 콘텐츠 생성 에이전트                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "title": "일상 속 편리함을 선사하는 AI 에이전트의 비밀",                                                     │
│    "content": "AI 에이전트는 현대 사회에서 보이지 않게 우리의 생활을 지원하는 중요한 도구입니다. 스마트폰에서   │
│  사용하는 음성 비서나 고객 문의에 답하는 챗봇, 그리고 온라인 쇼핑에서의 추천 시스템 등은 모두 AI 에이전트의     │
│  역할을 맡고 있습니다. 이들은 우리의 명령을 이해하고 최적의 해결책을 찾아주는 인공지능 시스템입니다. AI         │
│  에이전트는 단순히 우리가 지시한 일을 수행하는 것에서 나아가, 사용자의 행동 패턴을 학습하고 이를 바탕으로       │
│  맞춤형 서비스를 제공합니다. 예를 들어, 음악 스트리밍 서비스는 사용자의 청취 기록을 분석해 취향에 맞는 음악을   │
│  추천해 줍니다.\n\nAI 에이전트의 발전은 기술의 미래를 열어주는 열쇠로 작용합니다. 자율주행차, 스마트홈 기기 등  │
│  다양한 분야에 응용되어 우리의 삶에 더 큰 변화를 가져올 것입니다. 이러한 변화는 비전공자에게도 복잡하게 느껴질  │
│  수 있지만, 본질적으로는 더 나은 삶을 위한 기술적 진보를 의미합니다. AI 에이전트는 이미 그 흐름에 한가운데에    │
│  있으며, 계속해서 진화할 것입니다."                                                                             │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  에이전트에 대한 블로그 제목과 본문을 작성하라. 본문은 200단어 이내로 구성하고, 비전공자도 이해할 수 있게       │
│  설명한다.                                                                                                      │
│  Agent:                                                                                                         │
│  블로그 콘텐츠 생성 에이전트                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  이전 Task에서 생성한 블로그 글을 바탕으로, 핵심 내용을 요약하고 주요 포인트를 정리하라. 경영진이 빠르게 읽을   │
│  수 있는 요약본 형식으로 작성한다.                                                                              │
│  Agent:                                                                                                         │
│  블로그 콘텐츠 생성 에이전트                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



╭────────────────────────────── Execution Traces ──────────────────────────────╮
│                                                                              │
│  🔍 Detailed execution traces are available!                                 │
│                                                                              │
│  View insights including:                                                    │
│    • Agent decision-making process                                           │
│    • Task execution flow and timing                                          │
│    • Tool usage details                                                      │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯
Would you like to view your execution traces? [y/N] (20s timeout): 



╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                                                              │
│  Info: Tracing has been disabled.                                            │
│                                                                              │
│  Your preference has been saved. Future Crew/Flow executions will not        │
│  collect traces.                                                             │
│                                                                              │
│  To enable tracing later, do any one of these:                               │
│  • Set tracing=True in your Crew/Flow code                                   │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file               │
│  • Run: crewai traces enable                                                 │
│                                                                              │
╰─────────────────────────

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  ea1ebf5a-10ef-4393-84f1-5cfe0a647043                                                                           │
│  Final Output: # 일상 속 편리함을 선사하는 AI 에이전트의 비밀 요약                                              │
│                                                                                                                 │
│  AI 에이전트는 우리의 일상 속 다양한 기술에 통합되어 유용한 도구로 작용하며, 사용자에게 맞춤형 서비스를         │
│  제공하고 있습니다. 이러한 AI 기술의 발전은 자율주행차와 스마트홈 기기 등으로 우리의 삶을 더 편리하고           │
│  효율적으로 변화시킬 것입니다.                                                                                  │
│                                                                                                                 │
│  **핵심 포인트:**                                                                                               │
│  - AI 에이전트는 음성 비서, 챗봇, 추천 시스템 등 다양한 형태로 우리의 생활을 지원합니다.                        │
│  - 명령을 이해하고 최적의 해결책을 제시하는 인공지능 시스템입니다.                                              │
│  - 사용자 행동 패턴을 학습하여 맞춤형 서비스를 제공합니다.                                                      │
│  - 기술의 발전은 더 나은 삶을 위한 변화를 가져옵니다.                                                           │
│  - AI 에이전트는 미래 기술 혁신의 중추적인 역할을 맡고 있습니다.                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [7]:
print("=== Crew Result ===")
print(result.raw)                             # 크루 단위 통합 결과(원문)

print("\n=== Blog Generation Output (Pydantic) ===")
print(write_blog_task.output.pydantic.model_dump())   # 블로그 원문(검증된 스키마)

print("\n=== Summary Output (Markdown) ===")
print(save_task.output.raw)                  # 요약본 마크다운 결과

=== Crew Result ===
# AI 에이전트: 일상 속 보이지 않는 조력자 요약

AI 에이전트는 다양한 형태로 우리의 일상에 스며들어 삶의 불편함을 줄이고 효율성을 높이는 데 기여합니다. 이들은 지속적인 학습을 통해 데이터 분석을 기반으로 사용자 경험을 개인화하며, 미래에는 더욱 다양한 분야에서 혁신을 이끌 전망입니다.

**핵심 포인트:**
- AI 에이전트는 일상 속에서 자연스럽게 자리잡아, 음성 비서와 챗봇 등으로 나타납니다.
- AI 시스템은 명령을 이해해 작업을 수행하고, 사용자와 상호작용합니다.
- 에이전트는 학습을 거쳐 더 지능적으로 진화하며, 맞춤형 서비스 제공을 가능하게 합니다.
- AI 에이전트의 발전은 자율주행차와 스마트홈 기기 등과 결합해 삶을 더욱 자동화하고 편리하게 할 것입니다.
- 이러한 기술은 복잡해 보일 수 있지만 결과적으로 우리의 삶을 개선하는 역할을 합니다.

=== Blog Generation Output (Pydantic) ===
{'title': 'AI 에이전트: 일상 속 보이지 않는 조력자', 'content': 'AI 에이전트는 우리의 일상 생활에 자연스럽게 녹아들어 불편함을 없애고 효율성을 높이고 있습니다. 스마트폰의 음성 비서, 고객 서비스 챗봇, 심지어 온라인 쇼핑 추천 시스템까지, 이들은 모두 다양한 형태의 AI 에이전트입니다. 그러면 AI 에이전트란 정확히 무엇일까요? 간단히 말해, AI 에이전트는 특정한 작업을 수행하도록 만들어진 인공지능 시스템입니다. 우리의 요청이나 명령을 이해하고, 이를 바탕으로 적절한 행동을 합니다.\n\n이러한 에이전트는 단순히 일을 처리하는 데 그치지 않고, 지속적인 학습을 통해 더욱 똑똑해집니다. 예를 들어, 음악 스트리밍 서비스는 사용자의 청취 패턴을 학습하여 맞춤형 재생 목록을 추천합니다. 이처럼 AI 에이전트는 데이터 분석을 통해 사용자 경험을 개인화시킵니다.\n\nAI 에이전트의 미래는 더욱 유망합니다. 자율주행차나 똑똑한 가정용 기기까지, 우리의 삶은 더욱 자

In [ ]:
# 도구

In [17]:
# RAG Tool

from crewai_tools import RagTool

# 1) RAG 도구 생성 (기본 설정)
rag_tool = RagTool(
    name="MyDocsRAG",
    description="내 문서를 기반으로 질문에 답하는 RAG 툴",
    summarize=True,          # 검색 결과를 한 번 요약해서 응답에 사용
    verbose=True,             # 내부 동작 로깅
)

# 2) 문서 추가 (벡터DB에 임베딩 + 저장)
rag_tool.add(data_type="file", path="./knowledge/KB주택시장리뷰_2026년 1월호.pdf")
# 웹페이지도 추가 가능
rag_tool.add(data_type="website", url="https://www.kbfg.com/kbresearch/report/reportList.do")

# 3) 질문하기
question = "서울 및 수도권 아파트 시장"
answer = rag_tool.run(question)

print("Q:", question)
print("A:", answer)

Using Tool: MyDocsRAG
Q: 서울 및 수도권 아파트 시장
A: Relevant Content:



Page 6:
 
 
 
 6
[전세시장] 2025년 주택 전세가격은 전년 대비 상승폭 축소 
 
■ 주택 전세가격은 12월 전월 대비 0.21%, 2025년 연간 1.01% 상승 
 2025년 하반기 들어 수도권과 비수도권 모두 상승폭이 확대 
∙ 수도권은 12월 전월 대비 0.36%, 2025년 연간 1.72% 상승. 비수도권도 12월 전월 대비 5개광
역시는 +0.30%, 기타지방은 +0.29%, 2025년 연간 각각 0.39%, 0.20% 오름 
 KB전세가격전망지수는 수도권뿐 아니라 비수도권에서도 상승 전망이 우세 
 
그림 9| 지역별 주택 전세가격지수 변동률 추이 
 
그림 10| 지역별 주택 전세가격지수 변동률(2025년) 
 
 
 
자료: KB국민은행 
 
자료: KB국민은행 
 
■ 수도권 중심의 규제 강화와 전세물량 부족으로 올해도 전세시장 불안은 지속될 가능성 
 신규 입주예정물량 감소, 규제 강화에 따른 매수 심리 냉각, 갭투자 금지에 따른 매물 감소 등으로 
수도권 전세시장 불안은 당분간 지속될 전망 
∙ 올해 신규 입주예정물량은 21만 호로 전년 대비 24.4% 감소했으며, 특히 수도권은 11.1만 호(전
년 대비 -19.0%)에 그칠 전망 
 
그림 11| 신규 입주예정물량과 전세가격 추이 
 
그림 12| 전세수급지수 추이 
 
 
 
자료: KB국민은행, 부동산114 
 
자료: KB국민은행 
-0.6
-0.3
0
0.3
0.6
'23.6
'23.11
'24.4
'24.9
'25.2
'25.7
'25.12
전국
수도권
5개광역시
기타지방
(전월대비, %)
1.0 
2.6 
1.6 
0.1 
1.1 
-1.2 
0.4 
-1.0 
3.5 
0.2 
-3
-2
-1
0
1
2
3
4
5
전국서울경기인천부산대구광주대전울산기타
지방
2024년
2025년
(전년대비, %)
-20
-10
0
10
20
30
40
6


In [18]:
# config 
config = {
    "vectordb": {
        "provider": "chromadb",
        "config": {
            "collection_name": "kb-collection"
        }
    },
    "embedding_model": {
        "provider": "openai",
        "config": {
            "model_name": "text-embedding-3-small", 
        }
    },
    "chunker": {
        "chunk_size": 300,
        "chunk_overlap": 50,
        "length_function": "len",
        "min_chunk_size": 0 
    }
}


In [ ]:
# 1) RAG 도구 정의 (앞에서 만든 config 사용)
rag_tool = RagTool(
    name="MyDocsRAG",
    description="KB 리포트를 기반으로 주택시장 질의에 답하는 RAG 툴",
    summarize=True,
    result_as_answer=True,
    verbose=True,
    config=config,
)

# KB 주택시장 리뷰 PDF 추가
rag_tool.add(
    data_type="file",
    path="./knowledge/KB주택시장리뷰_2026년 1월호.pdf",
)

# KB 연구보고서 리스트 페이지 추가
rag_tool.add(
    data_type="website",
    url="https://www.kbfg.com/kbresearch/report/reportList.do",
)

# 2) 에이전트 정의
housing_research_agent = Agent(
    role="주택시장 리서치 에이전트",
    goal=(
        "KB 주택시장 리뷰 및 관련 자료를 기반으로 "
        "한국 주택시장 동향과 리스크를 정리한다."
    ),
    backstory=(
        "KB금융지주 리서치센터에서 근무해 온 시니어 애널리스트로, "
        "보고서와 데이터에 근거해 인사이트를 제공한다."
    ),
    tools=[rag_tool],
    verbose=True,
)

responder_agent = Agent(
    role="리포트 작성 에이전트",
    goal="검색된 내용을 바탕으로 이해하기 쉬운 한국어 분석 리포트를 작성한다.",
    backstory="경제 리포트와 시장 분석 문서를 다수 작성해 온 애널리스트이다.",
    verbose=True,
)

# 3) 태스크 정의
research_task = Task(
    description=(
        "다음 질문에 대해 KB 주택시장 리뷰 및 연결된 자료를 먼저 탐색하라.\n"
        "질문: {question}\n\n"
        "요구사항:\n"
        "1) 반드시 RAG 툴(MyDocsRAG)을 사용해 관련 문서를 검색한다.\n"
        "2) 후속 태스크가 이해하기 쉽도록 핵심 내용을 정리해 둔다.\n"
    ),
    expected_output=(
        "다음 정보를 포함하는 한국어 요약 노트를 작성한다.\n\n"
        "1. 관련성이 높은 문단/페이지에서 인용한 핵심 문장 정리 (3~7개 불릿)\n"
        "2. 각 인용 내용의 출처 표기 (예: 'KB 주택시장 리뷰 2026년 1월호, p.12')\n"
    ),
    agent=housing_research_agent,
    tools=[rag_tool],
    verbose=True,
)

housing_analysis_task = Task(
    description=(
        "다음 질문에 대해 KB 주택시장 리뷰 및 연결된 자료를 우선적으로 활용해 답변하라.\n"
        "질문: {question}\n\n"
        "요구사항:\n"
        "1) 보고서에 명시된 내용과 수치를 중심으로 정리한다.\n"
        "2) 인용한 내용이 어느 시기(연도/월)의 리뷰인지 간단히 언급한다.\n"
        "3) 문서에서 찾기 어려운 내용은 '해당 문서에서 명확히 찾기 어렵다'고 표시한다.\n"
    ),
    expected_output=(
        "다음 구조의 한국어 분석 리포트를 작성한다.\n\n"
        "1. 한 줄 요약\n"
        "2. 핵심 내용 정리 (3~5개 불릿)\n"
        "3. 세부 설명 (문단 형식)\n"
        "4. 출처·참고 (예: 'KB 주택시장 리뷰 2026년 1월호 기준')"
    ),
    agent=responder_agent,
    context=[research_task],
    verbose=True,
)

# 4) Crew 구성
housing_crew = Crew(
    agents=[housing_research_agent, responder_agent],
    tasks=[research_task, housing_analysis_task],
    process=Process.sequential,
    verbose=True,
)

# 5) 실행 예시
if __name__ == "__main__":
    question = "2026년 1월 기준 서울 및 수도권 아파트 시장의 주요 특징과 리스크는 무엇인가?"

    result = housing_crew.kickoff(inputs={"question": question})

    print("\n=== Crew 통합 결과(result.raw) ===")
    print(result.raw)

    print("\n=== Research Task 출력 ===")
    print(research_task.output.raw)

    print("\n=== 분석 태스크(housing_analysis_task) 원문 출력 ===")
    print(housing_analysis_task.output.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  ed49d82b-f533-4f2c-b6c1-d9568d808ce1                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 다음 질문에 대해 KB 주택시장 리뷰 및 연결된 자료를 먼저 탐색하라.                                        │
│  질문: 2026년 1월 기준 서울 및 수도권 아파트 시장의 주요 특징과 리스크는 무엇인가?                              │
│                                                                                                                 │
│  요구사항:                                                                                                      │
│  1) 반드시 RAG 툴(MyDocsRAG)을 사용해 관련 문서를 검색한다.                                                     │
│  2) 후속 태스크가 이해하기 쉽도록 핵심 내용을 정리해 둔다.                                                      │
│                                                                                                                 │
│  ID: fa9dcfc2-980b-434a-b1e5-57b3fb519ad1                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 주택시장 리서치 에이전트                                                                                │
│                                                                                                                 │
│  Task: 다음 질문에 대해 KB 주택시장 리뷰 및 연결된 자료를 먼저 탐색하라.                                        │
│  질문: 2026년 1월 기준 서울 및 수도권 아파트 시장의 주요 특징과 리스크는 무엇인가?                              │
│                                                                                                                 │
│  요구사항:                                                                                                      │
│  1) 반드시 RAG 툴(MyDocsRAG)을 사용해 관련 문서를 검색한다.                                                     │
│  2) 후속 태스크가 이해하기 쉽도록 핵심 내용을 정리해 둔다.                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#2) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: MyDocsRAG                                                                                                │
│  Args: {"query": "2026\ub144 1\uc6d4 \uc11c\uc6b8 \uc218\ub3c4\uad8c \uc544\ud30c\ud2b8 \uc2dc\uc7a5            │
│  \ud2b9\uc9d5 \ub9ac\uc2a4\ud06c", "similarity_threshold": 0.7, "limit": 5}                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 주택시장 리서치 에이전트                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Relevant Content:                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│  Page 6:                                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│   6                                                                                                             │
│  [전세시장] 2025년 주택 전세가격은 전년 대비 상승폭 축소                                                        │
│                                                                                                                 │
│  ■ 주택 전세가격은 12월 전월 대비 0.21%, 2025년 연간 1.01% 상승                                                 │
│   2025년 하반기 들어 수도권과 비수도권 모두 상승폭이 확대                                                      │
│  ∙ 수도권은 12월 전월 대비 0.36%, 2025년 연간 1.72% 상승. 비수도권도 12월 전월 대비 5개광                       │
│  역시는 +0.30%, 기타지방은 +0.29%, 2025년 연간 각각 0.39%, 0.20% 오름                                           │
│   KB전세가격전망지수는 수도권뿐 아니라 비수도권에서도 상승 전망이 우세                                         │
│                                                                                                                 │
│  그림 9| 지역별 주택 전세가격지수 변동률 추이                                                                   │
│                                                                                                                 │
│  그림 10| 지역별 주택 전세가격지수 변동률(2025년)                                                               │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│  자료: KB국민은행                                                                                               │
│                                                                                                                 │
│  자료: KB국민은행                                                                                               │
│                                                                                                                 │
│  ■ 수도권 중심의 규제 강화와 전세물량 부족으로 올해도 전세시장 불안은 지속될 가능성                             │
│   신규 입주예정물량 감소, 규제 강화에 따른 매수 심리 냉각, 갭투자 금지에 따른 매물 감소 등으로                 │
│  수도권 전세시장 불안은 당분간 지속될 전망                                                                      │
│  ∙ 올해 신규 입주예정물량은 21만 호로 전년 대비 24.4% 감소했으며, 특히 수도권은 11.1만 호(전                    │
│  년 대비 -19.0%)에 그칠 전망                                                                                    │
│                            

...

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): 



╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                                                              │
│  Info: Tracing has been disabled.                                            │
│                                                                              │
│  Your preference has been saved. Future Crew/Flow executions will not        │
│  collect traces.                                                             │
│                                                                              │
│  To enable tracing later, do any one of these:                               │
│  • Set tracing=True in your Crew/Flow code                                   │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file               │
│  • Run: crewai traces enable                                                 │
│                                                                              │
╰─────────────────────────

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  ed49d82b-f533-4f2c-b6c1-d9568d808ce1                                                                           │
│  Final Output: 1. 한 줄 요약                                                                                    │
│  2026년 1월 기준 서울 및 수도권 아파트 시장은 2025년에 강한 가격 상승세를 보였으나, 정부 규제 강화로 거래       │
│  위축과 매수심리 둔화가 뚜렷하며 전세 및 월세 시장에서는 물량 부족과 임대료 상승 리스크가 지속되고 있다.        │
│                                                                                                                 │
│  2. 핵심 내용 정리                                                                                              │
│  - 2025년 서울 아파트 매매가격은 연간 +7.44%, 수도권은 +3.04% 상승했으나 10월 정부 규제 강화 이후 상승폭과      │
│  거래량이 빠르게 둔화                                                                                           │
│  - 11월 서울 아파트 매매 거래량은 전월 대비 절반 이상 감소, 수도권 규제 지역 중심으로 거래 위축 심화            │
│  - 수도권 전셋값은 2025년 연간 +1.72% 상승, 전세 물량 부족과 규제로 전세시장 불안 지속 전망                     │
│  - 월세 시장은 전세가격 상승 영향과 1·2인 가구 증가로 수도권 아파트 월세가격이 12월 기준 연평균 8.0% 급등하며   │
│  월세 비중도 47.8%로 확대                                                                                       │
│  - 수도권 신규 입주 물량 감소(전년 대비 약 19% 감소), 전세 공급 부족 리스크 및 대출 규제 강화가 시장            │
│  불안요인으로 작용                                                                                              │
│                                                                                                                 │
│  3. 세부 설명                                                                                                   │
│  2025년 한 해 동안 서울 아파트 매매가격은 7.44% 상승하며 2021년 이후 최고 상승률을 기록하였고, 수도권 전체도    │
│  3.04% 상승하는 등 강세를 보였다. 특히 강남 3구(강남구 +16.92%, 송파구 +16.84%, 서초구 +13.0%),                 │
│  성동구(+18.51%), 분당(+19.11%), 과천(+21.99%) 등 인기지역 중심으로 가격 상승이 두드러졌다. 그러나 2025년 10월  │
│  15일 정부가 규제 지역을 확대하고 대출 규제를 강화한 이후, 서울 및 수도권 주요 아파트 가격 상승폭은 급격히      │
│  둔화되고 매수 심리도 약화되었다. 예를 들어, 서울 아파트 주간 매매가격 변동률은 10월 중 0.68%에서 1월 5일 기준  │
│  0.19%로 감소했다.                                                                                              │
│                                                                                                                 │
│  거래량도 영향을 크게 받았다. 11월 서울의 아파트 거래량은 약 7,570건으로 전월 대비 절반 이상 줄었고, 수도권     │
│  규제 지역 거래는 30% 감소했다. 전반적으로 매수 우위 지수와 매매 거래지수 모두 낮은 수준을 유지하고 있으며,     │
│  12월 거래량은 반등하는 조짐을 보이나 여전히 회복은 미미한 상황이다.                                            │
│                                                                                                                 │
│  전세시장에서는 2025년 수도권 전세가격이 연간 1.72%, 서울도 전월 대비 0.36% 상승하며 전세 가격 상승세가         │
│  확대됐다. 신규 입주 물량은 2025년 약 21만 호로 전년 대비 24.4% 감소했으며, 수도권은 11.1만 호에 그쳐 전년      │
│  대비 19.0% 감소했다. 이로 인해 전세 공급 부족이 심화되고, 전세 시장의 불안 요인이 지속되고 있다. 이에 따라 갭  │
│  투자 금지 및 매수 심리 위축 등 규제가 전세 시장에도 영향을 미치고 있다.                                        │
│                                                                                                                 │
│  월세 시장은 이러한 전세 물량 부족 현상과 1·2인 가구의 꾸준한 수도권 유입으로 인해 월세 가격이 큰 폭으로        │
│  상승했다. 2025년 12월 기준 수도권 아파트 월세가격은 전년 동월 대비 8.0% 상승했으며, 서울 아파트 전월세 매물    │
│  중 월세 비중은 47.8%까지 확대되었다. 이는 전월세 시장 내 임대차 구조 변화가 가속화되고 있음을 시사한다.        │
│                                                                                                                

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [ ]:
# SerperDev Tool

In [20]:
from dotenv import load_dotenv
load_dotenv()  # 현재 경로의 .env 파일을 불러와 환경변수로 등록

# os.environ["SERPER_API_KEY"] = "여기에_본인_API_키"

True

In [21]:
from crewai_tools import SerperDevTool

tool = SerperDevTool(n_results=1)
response = tool.run(search_query="한국 금리 전망")
print(response)

Using Tool: Search the internet with Serper
{'searchParameters': {'q': '한국 금리 전망', 'type': 'search', 'num': 1, 'engine': 'google'}, 'organic': [{'title': '한국 이자율 | 1999-2026 데이터 | 2027-2028 예상', 'link': 'https://ko.tradingeconomics.com/south-korea/interest-rate', 'snippet': '한국은행은 올해 헤드라인 인플레이션을 2.0%, 2026년에는 1.9%로 예상하고, 코어 인플레이션은 두 해 동안 모두 1.9%로 유지될 것으로 예상된다.', 'position': 1}], 'credits': 1}


In [22]:
tool = SerperDevTool(
    country="kr",
    locale="ko",
    n_results=1
)
print(tool.run(search_query="부동산 시장 2026 전망"))

Using Tool: Search the internet with Serper
{'searchParameters': {'q': '부동산 시장 2026 전망', 'type': 'search', 'gl': 'kr', 'hl': 'ko', 'num': 1, 'engine': 'google'}, 'organic': [{'title': '2026 국내 상업용 부동산 시장 전망', 'link': 'https://www.cbrekorea.com/insights/reports/2026-%EA%B5%AD%EB%82%B4-%EC%83%81%EC%97%85%EC%9A%A9-%EB%B6%80%EB%8F%99%EC%82%B0-%EC%8B%9C%EC%9E%A5-%EC%A0%84%EB%A7%9D', 'snippet': '2026년은 도심(CBD)을 중심으로 대형 자산 공급이 재개되며 팬데믹 이전 수준의 공급 물량으로 회귀하는 시기이다. 공실률은 소폭 상승하겠지만, 여전히 ...', 'position': 1}], 'credits': 1}


In [ ]:
# 1) SerperDevTool 정의 (웹 검색 전용 도구)
serper_tool = SerperDevTool(
    n_results=5,  # 상위 5개 결과만 사용
)

# 2) 에이전트 정의
# 2-1) 웹에서 자료를 찾는 리서치 에이전트
real_estate_researcher = Agent(
    role="부동산 리서치 에이전트",
    goal=(
        "웹 검색을 통해 최신 한국 부동산 시장 동향과 전망을 찾아 "
        "핵심 정보만 정리한다."
    ),
    backstory=(
        "각종 뉴스, 리포트, 칼럼을 분석해 요약해 온 "
        "온라인 리서치 전문가이다."
    ),
    tools=[serper_tool],  # ✅ SerperDevTool 연결
    verbose=True,
)

# 2-2) 리포트 형태로 정리하는 에이전트
report_writer = Agent(
    role="부동산 리포트 작성 에이전트",
    goal="수집된 정보를 바탕으로 이해하기 쉬운 한국어 시장 전망 리포트를 작성한다.",
    backstory="경제/부동산 관련 리포트를 다수 작성해 온 분석가이다.",
    verbose=True,
)

# 3) 태스크 정의
# 3-1) 웹에서 정보 수집
search_task = Task(
    description=(
        "다음 질문에 대해 SerperDevTool을 사용해 웹을 검색하고, "
        "관련성이 높은 기사와 리포트를 찾아 요약하라.\n"
        "질문: {question}\n\n"
        "요구사항:\n"
        "1) 최근 1~2년 내 기사/보고서를 우선적으로 참고한다.\n"
        "2) 서로 다른 출처(언론사, 리포트)를 최소 2개 이상 포함한다.\n"
    ),
    expected_output=(
        "다음 형식의 요약 노트를 작성한다.\n\n"
        "1. 참고한 주요 기사/리포트 목록 (제목 · 출처 · URL)\n"
        "2. 기사/리포트에서 공통적으로 언급하는 핵심 키워드 3~5개\n"
        "3. 각 키워드에 대한 한두 문장 요약"
    ),
    agent=real_estate_researcher,
    tools=[serper_tool],
    verbose=True,
)

# 3-2) 수집된 내용을 바탕으로 전망 리포트 작성
analysis_task = Task(
    description=(
        "이전 태스크에서 정리한 웹 검색 요약을 바탕으로 "
        "한국 부동산 시장 전망에 대해 분석 리포트를 작성하라.\n"
        "질문: {question}\n\n"
        "요구사항:\n"
        "1) 단기(1년 이내)와 중기(3년 이내) 전망을 나누어 서술한다.\n"
        "2) 상승/하락 요인과 리스크 요인을 구분해 정리한다.\n"
        "3) 과도한 확신 표현은 피하고, '가능성이 높다/낮다' 수준으로 표현한다.\n"
    ),
    expected_output=(
        "다음 구조의 한국어 리포트를 작성한다.\n\n"
        "1. 한 줄 요약\n"
        "2. 단기 전망 (1년 이내)\n"
        "3. 중기 전망 (3년 이내)\n"
        "4. 주요 상승 요인\n"
        "5. 주요 하락·리스크 요인\n"
        "6. 참고: 사용한 기사/리포트의 간단한 출처 요약"
    ),
    agent=report_writer,
    context=[search_task],  # ✅ 검색 결과를 컨텍스트로 활용
    verbose=True,
)

# 4) Crew 구성
real_estate_crew = Crew(
    agents=[real_estate_researcher, report_writer],
    tasks=[search_task, analysis_task],
    process=Process.sequential,  # 검색 → 분석 순차 실행
    verbose=True,
)

# 5) 실행 예시
if __name__ == "__main__":
    question = "2026년 향후 한국 부동산 시장 전망과 주요 리스크는 무엇인가?"

    result = real_estate_crew.kickoff(inputs={"question": question})

    print("\n=== Crew 통합 결과(result.raw) ===")
    print(result.raw)

    print("\n=== 검색 태스크(search_task) 출력 ===")
    print(search_task.output.raw)


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  a1558d5b-8547-4b82-8cf4-3a45817f4edd                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 다음 질문에 대해 SerperDevTool을 사용해 웹을 검색하고, 관련성이 높은 기사와 리포트를 찾아 요약하라.      │
│  질문: 2026년 향후 한국 부동산 시장 전망과 주요 리스크는 무엇인가?                                              │
│                                                                                                                 │
│  요구사항:                                                                                                      │
│  1) 최근 1~2년 내 기사/보고서를 우선적으로 참고한다.                                                            │
│  2) 서로 다른 출처(언론사, 리포트)를 최소 2개 이상 포함한다.                                                    │
│                                                                                                                 │
│  ID: f6824e2f-a89d-4cbe-8776-5d59919c3127                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 부동산 리서치 에이전트                                                                                  │
│                                                                                                                 │
│  Task: 다음 질문에 대해 SerperDevTool을 사용해 웹을 검색하고, 관련성이 높은 기사와 리포트를 찾아 요약하라.      │
│  질문: 2026년 향후 한국 부동산 시장 전망과 주요 리스크는 무엇인가?                                              │
│                                                                                                                 │
│  요구사항:                                                                                                      │
│  1) 최근 1~2년 내 기사/보고서를 우선적으로 참고한다.                                                            │
│  2) 서로 다른 출처(언론사, 리포트)를 최소 2개 이상 포함한다.                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: Search the internet with Serper                                                                          │
│  Args: {"search_query": "2026\ub144 \ud55c\uad6d \ubd80\ub3d9\uc0b0 \uc2dc\uc7a5 \uc804\ub9dd \uc8fc\uc694      │
│  \ub9ac\uc2a4\ud06c 2022 \uc774\ud6c4"}                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  {'searchParameters': {'q': '2026년 한국 부동산 시장 전망 주요 리스크 2022 이후', 'type': 'search', 'num': 5,   │
│  'engine': 'google'}, 'organic': [{'title': '장밋빛 전망이 넘치는 2026년 부동산시장- 우리가 주의해야 ...',      │
│  'link': 'https://blog.naver.com/audi72/224121259438?viewType=pc', 'snippet': '2026년 부동산시장 참여자들이     │
│  조심해야 할 테일리스크입니다. 이런 문제 ... 미국 연준은 금리를 신속히 연속적으로 인하할 것입니다. 그리고       │
│  그것이 한국 ...', 'position': 1}, {'title': '2026년 부동산 시장에 변화의 기류', 'link':                        │
│  'http://www.koreatimes.com/article/20260114/1597084', 'snippet': '2022년 이후 급격한 금리 인상은 주택 시장을   │
│  빠르게 냉각시켰다. 구매 ... Realtor.com과 Zillow 역시 2026년을 급격한 반등보다는 점진적인 회복과 안정의 ...',  │
│  'position': 2}, {'title': '2026년 부동산 시장 집값 전망, 부동산 2월부터 무서울 ...', 'link':                   │
│  'https://www.youtube.com/watch?v=KZfpgIhvcoo', 'snippet': '부동산 하락 전망 어떻게 합니까? 2026년 집값         │
│  폭등하는 진짜 이유 (김인만 소장 3부) - 촬영일시: 1월 2일 김작가 저서 『머니 트렌드 2026』 도서 ...',           │
│  'position': 3}, {'title': "대출 안 나오는 '비싼 집'이 더 올랐다…2026년엔 다를까?", 'link':                     │
│  'https://www.hani.co.kr/arti/economy/property/1237835.html', 'snippet': '이후 실제로 전개된 2025년 부동산      │
│  시장 상황은, 우여곡절을 겪으면서 연구원의 예측치보다 더 큰 폭으로 서울·수도권 집값이 상승했습니다.             │
│  한국부동산원 ...', 'position': 4}, {'title': '[결산&전망] 2026년 아파트 경매시장 전망과 전략', 'link':         │
│  'https://m.r114.com/?_c=research&_m=research_detail&bno=200&gno=7&num=8232', 'snippet': '다만 아파트 경매      │
│  진행건수는 여전히 증가세를 보이고 있다. 2025년 10월 기준 약 3만건이 진행되었으며, 2022년 이후로는 해마다 1만   │
│  건 안팎의 증가 폭 ...', 'position': 5}], 'credits': 1}                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  이전 태스크에서 정리한 웹 검색 요약을 바탕으로 한국 부동산 시장 전망에 대해 분석 리포트를 작성하라.            │
│  질문: 2026년 향후 한국 부동산 시장 전망과 주요 리스크는 무엇인가?                                              │
│                                                                                                                 │
│  요구사항:                                                                                                      │
│  1) 단기(1년 이내)와 중기(3년 이내) 전망을 나누어 서술한다.                                                     │
│  2) 상승/하락 요인과 리스크 요인을 구분해 정리한다.                                                             │
│  3) 과도한 확신 표현은 피하고, '가능성이 높다/낮다' 수준으로 표현한다.                                          │
│                                                                                                                 │
│  Agent:                                                                                                         │
│  부동산 리포트 작성 에이전트                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


=== Crew 통합 결과(result.raw) ===
한국 부동산 시장 2026년 전망 및 주요 리스크 분석 리포트

1. 한 줄 요약  
2026년 한국 부동산 시장은 주택 공급 부족과 금융 규제 강화가 맞물리며 지역별 양극화와 전세 시장 불안이 심화될 가능성이 높아, 단기에는 조정 국면이 계속되나 중기적으로는 일부 지역에서 안정 내지 완만한 회복이 기대된다.

2. 단기 전망 (1년 이내)  
단기적으로는 수도권을 중심으로 신규 입주 물량이 감소하면서 공급 부족 현상이 심화되고, 강도 높은 대출 규제와 DSR 확대 적용으로 가계의 주택 구입 여력이 제한될 전망이다. 이에 따라 주택 거래는 다소 위축되면서 가격 상승 폭이 둔화될 가능성이 크다. 그러나 전세 공급 부족으로 전세 가격은 상승세가 이어질 것으로 보이며, 주거 안정성에 대한 부담이 커질 수 있다. 정부의 규제 정책도 지속 시행되어 부동산 시장에 긴장감이 유지될 것으로 예측된다.

3. 중기 전망 (3년 이내)  
중기적으로는 수도권 일부 지역에서 새 주택 입주가 점차 증가하면서 주택 공급 부족 현상 완화가 점진적으로 기대된다. 다만 지방은 인구 감소 및 경제 저성장 추세가 심화되어 부동산 시장 침체가 지속될 가능성이 높다. 이로 인한 지역별 양극화 심화가 한국 부동산 시장의 큰 과제로 남을 전망이다. 대출 규제 완화 여부와 정부 정책 변화가 중요한 변수로 작용하며, 투자 심리 회복에 일부 긍정적 영향을 줄 수 있다. 전체적으로 중기 내 시장은 안정적 흐름을 찾으려는 모습을 보일 가능성이 높으나, 불확실성과 리스크도 병존할 것으로 분석된다.

4. 주요 상승 요인  
- 수도권 중심 신규 주택 입주 감소에 따른 공급 부족 심화로 가격 상승 압력 증가 가능성  
- 전세 공급 부족 지속에 따른 전세가 상승 및 임대 시장 불안 가중 가능성  
- 일부 재건축·재개발 사업 추진과 정부의 지역별 맞춤형 주택 정책이 특정 지역의 회복에 기여할 가능성

5. 주요 하락·리스크 요인  
- 강도 높은 대출 규

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  a1558d5b-8547-4b82-8cf4-3a45817f4edd                                                                           │
│  Final Output: 한국 부동산 시장 2026년 전망 및 주요 리스크 분석 리포트                                          │
│                                                                                                                 │
│  1. 한 줄 요약                                                                                                  │
│  2026년 한국 부동산 시장은 주택 공급 부족과 금융 규제 강화가 맞물리며 지역별 양극화와 전세 시장 불안이 심화될   │
│  가능성이 높아, 단기에는 조정 국면이 계속되나 중기적으로는 일부 지역에서 안정 내지 완만한 회복이 기대된다.      │
│                                                                                                                 │
│  2. 단기 전망 (1년 이내)                                                                                        │
│  단기적으로는 수도권을 중심으로 신규 입주 물량이 감소하면서 공급 부족 현상이 심화되고, 강도 높은 대출 규제와    │
│  DSR 확대 적용으로 가계의 주택 구입 여력이 제한될 전망이다. 이에 따라 주택 거래는 다소 위축되면서 가격 상승     │
│  폭이 둔화될 가능성이 크다. 그러나 전세 공급 부족으로 전세 가격은 상승세가 이어질 것으로 보이며, 주거 안정성에  │
│  대한 부담이 커질 수 있다. 정부의 규제 정책도 지속 시행되어 부동산 시장에 긴장감이 유지될 것으로 예측된다.      │
│                                                                                                                 │
│  3. 중기 전망 (3년 이내)                                                                                        │
│  중기적으로는 수도권 일부 지역에서 새 주택 입주가 점차 증가하면서 주택 공급 부족 현상 완화가 점진적으로         │
│  기대된다. 다만 지방은 인구 감소 및 경제 저성장 추세가 심화되어 부동산 시장 침체가 지속될 가능성이 높다. 이로   │
│  인한 지역별 양극화 심화가 한국 부동산 시장의 큰 과제로 남을 전망이다. 대출 규제 완화 여부와 정부 정책 변화가   │
│  중요한 변수로 작용하며, 투자 심리 회복에 일부 긍정적 영향을 줄 수 있다. 전체적으로 중기 내 시장은 안정적       │
│  흐름을 찾으려는 모습을 보일 가능성이 높으나, 불확실성과 리스크도 병존할 것으로 분석된다.                       │
│                                                                                                                 │
│  4. 주요 상승 요인                                                                                              │
│  - 수도권 중심 신규 주택 입주 감소에 따른 공급 부족 심화로 가격 상승 압력 증가 가능성                           │
│  - 전세 공급 부족 지속에 따른 전세가 상승 및 임대 시장 불안 가중 가능성                                         │
│  - 일부 재건축·재개발 사업 추진과 정부의 지역별 맞춤형 주택 정책이 특정 지역의 회복에 기여할 가능성             │
│                                                                                                                 │
│  5. 주요 하락·리스크 요인                                                                                       │
│  - 강도 높은 대출 규제와 금융 환경 악화로 주택 구입 여력 저하 및 투자 심리 위축 가능성                          │
│  - 지방 지역의 인구 감소 및 경제 침체 심화로 인한 부동산 시장 침체 및 가격 하락 위험 증가                       │
│  - 정부의 고가 주택 및 투자 수요에 대한 규제 강화 지속으로 매수세 약화 가능성                                   │
│  - 지역별 양극화 심화에 따른 시장 불균형 심화 및 전반적 부동산 시장 불안 요소 존재                              │
│                                                                                                                 │
│  6. 참고: 사용한 기사/리포트의 간단한 출처 요약                                                                 │
│  본 리포트는 KB국민은행의 “KB주택시장 리뷰-2026년                                                               │
│  1월호”(https://www.kbfg.com/kbresearch/report/reportView.do?reportId=2000555)와 한국개발연구원(KDI)의 “2026년  │
│  주택시장 전망”(https://eiec.kdi.re.kr/policy/domesticView.do?ac=0000201795) 두 보고서를 참고하여 작성하였다.   │
│  양 기관은 공급 부족, 대출 규제, 지역 양극화, 전세 불안, 정부 정책 변화 등 공통 키워드를 중심으로 2026년        │
│  부동산 

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



╭────────────────────────────── Execution Traces ──────────────────────────────╮
│                                                                              │
│  🔍 Detailed execution traces are available!                                 │
│                                                                              │
│  View insights including:                                                    │
│    • Agent decision-making process                                           │
│    • Task execution flow and timing                                          │
│    • Tool usage details                                                      │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯
Would you like to view your execution traces? [y/N] (20s timeout): 

╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                      

In [ ]:
# 지식 

In [ ]:
# Create Custom Tool

from typing import Type
from pydantic import BaseModel, Field
from crewai.tools import BaseTool

# 1) 입력 스키마 정의 (Pydantic 기반)
class CurrencyInput(BaseModel):
    amount: float = Field(..., description="변환할 금액(숫자)")
    rate: float = Field(..., description="적용할 환율(예: 1325.5)")

# 2) BaseTool 상속하여 사용자 정의 도구 만들기
class CurrencyConverterTool(BaseTool):
    name: str = "환율 변환 도구"
    description: str = "주어진 금액을 환율에 따라 변환하여 반환합니다."
    args_schema: Type[BaseModel] = CurrencyInput

    def _run(self, amount: float, rate: float) -> str:
        """실행 로직"""
        result = amount * rate
        return f"{result:,.2f} 원"
    
tool = CurrencyConverterTool()

result = tool.run(amount=100, rate=1325.5)
print(result)


Using Tool: 환율 변환 도구
132,550.00 원


In [20]:
from crewai.tools import tool

# tool 데코레이터를 사용해 간단하게 사용자 정의 도구 만들기
@tool("환율 변환 도구")
def currency_converter(amount: float, rate: float) -> str:
    """주어진 금액(amount)에 환율(rate)을 적용하여 원화(KRW) 금액을 계산합니다."""
    result = amount * rate
    return f"{result:,.2f} 원"


# 실행 테스트
print(currency_converter.run(amount=100, rate=1325.5))

Using Tool: 환율 변환 도구
132,550.00 원


In [ ]:
from crewai.knowledge.source.string_knowledge_source import StringKnowledgeSource

# 1) 문자열 기반 Knowledge 생성
content = "사용자의 이름은 홍길동이고, 30살이며 서울에 거주한다."
string_source = StringKnowledgeSource(content=content)

# 2) 에이전트 정의
agent = Agent(
    role="사용자 정보 전문가",
    goal="사용자에 대한 질문에 정확히 답변한다.",
    backstory="사람의 프로필과 선호를 잘 이해하는 분석가이다.",
    verbose=True
)

task = Task(
    description="다음 사용자 정보 질문에 답하라: {question}",
    expected_output="질문에 대한 간단한 한국어 답변.",
    agent=agent,
)

# 3) 크루 생성 시 knowledge_sources에 등록
crew = Crew(
    agents=[agent],
    tasks=[task],
    process=Process.sequential,
    knowledge_sources=[string_source],
)

result = crew.kickoff(
    inputs={"question": "사용자의 이름과 사는 도시와 나이는 얼마인가?"}
)

print(result.raw)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 사용자 정보 전문가                                                                                      │
│                                                                                                                 │
│  Task: 다음 사용자 정보 질문에 답하라: 사용자의 이름과 사는 도시와 나이는 얼마인가?                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 사용자 정보 전문가                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  사용자의 이름은 홍길동이고, 나이는 30살이며 서울에 거주합니다.                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

사용자의 이름은 홍길동이고, 나이는 30살이며 서울에 거주합니다.


╭────────────────────────────── Execution Traces ──────────────────────────────╮
│                                                                              │
│  🔍 Detailed execution traces are available!                                 │
│                                                                              │
│  View insights including:                                                    │
│    • Agent decision-making process                                           │
│    • Task execution flow and timing                                          │
│    • Tool usage details                                                      │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯
Would you like to view your execution traces? [y/N] (20s timeout): 



╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                                                              │
│  Info: Tracing has been disabled.                                            │
│                                                                              │
│  Your preference has been saved. Future Crew/Flow executions will not        │
│  collect traces.                                                             │
│                                                                              │
│  To enable tracing later, do any one of these:                               │
│  • Set tracing=True in your Crew/Flow code                                   │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file               │
│  • Run: crewai traces enable                                                 │
│                                                                              │
╰─────────────────────────

In [33]:
# 에이전트 레벨 Knowledge만 초기화
# crew.reset_memories(command_type='agent_knowledge')

# 크루 전체 Knowledge를 포함해 모두 초기화
crew.reset_memories(command_type='knowledge')

In [23]:
# PDFKnowledgeSource

from crewai.knowledge.source.pdf_knowledge_source import PDFKnowledgeSource

pdf_source = PDFKnowledgeSource(
    file_paths=["./KB주택시장리뷰_2026년 1월호.pdf", 
                "./KB주택시장리뷰_2025년 12월호.pdf"]
)

In [24]:
from crewai.knowledge.knowledge_config import KnowledgeConfig

knowledge_config = KnowledgeConfig(
    results_limit=10,     # 최대 10개의 문서/청크만 활용
    score_threshold=0.5   # 관련성 점수 0.5 이상만 사용
)

agent = Agent(
    role="리서치 에이전트",
    goal="관련 문서를 기반으로 요약을 제공한다.",
    backstory="문서를 잘 읽고 핵심만 정리하는 분석가이다.",
    knowledge_config=knowledge_config,
)


In [ ]:
# 1) PDF KnowledgeSource 등록
pdf_source = PDFKnowledgeSource(
    file_paths=[
        "KB주택시장리뷰_2026년 1월호.pdf",
        "KB주택시장리뷰_2025년 12월호.pdf"
    ]
)

# 2) KnowledgeConfig 설정
knowledge_config = KnowledgeConfig(
    results_limit=10,      # 최대 10개의 청크만 활용
    score_threshold=0.5    # 유사도 0.5 이상만 사용
)

# 3) 에이전트 정의 (KnowledgeConfig 적용)
agent = Agent(
    role="리서치 에이전트",
    goal="KB 주택시장 리뷰 문서를 기반으로 질문에 답변한다.",
    backstory="주택시장 리포트를 잘 읽고 해석하는 분석가이다.",
    knowledge_config=knowledge_config,
)

# 4) Q&A 태스크 정의
task = Task(
    description=(
        "KB 주택시장 리뷰 문서를 기반으로 다음 질문에 답하라:\n"
        "{question}"
    ),
    expected_output="질문에 대한 구체적이고 간단한 한국어 답변.",
    agent=agent,
)

# 5) 크루 생성
crew = Crew(
    agents=[agent],
    tasks=[task],
    knowledge_sources=[pdf_source],
    process=Process.sequential,
)

# 6) 실행: 실제 질문 넣기
result = crew.kickoff(
    inputs={
        "question": "2026년 1월호와 2025년 12월호에서 공통으로 강조하는 시장 리스크는 무엇인가?"
    }
)
print(result.raw)


2026년 1월호와 2025년 12월호 KB 주택시장 리뷰 모두 공통적으로 다음과 같은 시장 리스크를 강조하고 있다. 첫째, 정부의 규제 강화(특히 수도권 규제 지역 확대와 대출 규제 강화)로 인해 매수 심리가 크게 위축되고 주택 매매 거래량이 급감하면서 시장 전반의 거래 위축 현상이 지속되고 있다는 점이다. 둘째, 전세시장에서는 갭 투자가 금지되고 신규 입주 물량이 감소하여 전세 공급 부족 현상이 계속되고 전세가격 상승 압력이 높아 전세시장 불안이 장기화될 우려가 있다는 점이다. 셋째, 주택담보대출과 전세자금대출 모두 감소하거나 증가폭이 크게 축소되고 있으며, 대출 금리가 상승하고 은행권 대출 관리가 강화돼 금융 여건이 악화되고 있어 주택 금융 부문에서의 리스크가 커지고 있다는 점이다. 요약하면, 규제 강화에 의한 매수 심리 위축과 거래량 감소, 전세 공급 부족에 따른 전세가격 불안, 그리고 대출 규제로 인한 주택금융 어려움이 두 달호에서 공통으로 강조하는 주요 시장 리스크이다.


╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): 



╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                                                              │
│  Info: Tracing has been disabled.                                            │
│                                                                              │
│  Your preference has been saved. Future Crew/Flow executions will not        │
│  collect traces.                                                             │
│                                                                              │
│  To enable tracing later, do any one of these:                               │
│  • Set tracing=True in your Crew/Flow code                                   │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file               │
│  • Run: crewai traces enable                                                 │
│                                                                              │
╰─────────────────────────

In [ ]:
# 추론 

In [ ]:
# 에이전트 정의 (KnowledgeConfig 적용)
agent = Agent(
    role="리서치 에이전트",
    goal="KB 주택시장 리뷰 문서를 기반으로 질문에 답변한다.",
    backstory="주택시장 리포트를 잘 읽고 해석하는 분석가이다.",
    knowledge_config=knowledge_config,
    reasoning=True,
    max_reasoning_attempts=3    
)

# 크루 생성
crew = Crew(
    agents=[agent],
    tasks=[task],
    knowledge_sources=[pdf_source],
    process=Process.sequential,
    verbose=True
)

# 실행: 실제 질문 넣기
result = crew.kickoff(
    inputs={
        "question": "2026년 1월호와 2025년 12월호에서 공통으로 강조하는 시장 리스크는 무엇인가?"
    }
)
print(result.raw)


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  f8f8a738-7f88-42c4-9b20-5f8634d96163                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔍 Knowledge Retrieval ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Knowledge Retrieval Started                                                                                    │
│  Status: Retrieving...                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: KB 주택시장 리뷰 문서를 기반으로 다음 질문에 답하라:                                                     │
│  2025년 12월호와 2026년 1월호에서 공통으로 강조하는 시장 리스크는 무엇인가?                                     │
│  ID: ac96d38a-01ad-4033-9e0a-1fdf6ac0e3ea                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 📚 Knowledge Retrieved ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Search Query:                                                                                                  │
│  KB 주택시장 리뷰 2025년 12월호와 2026년 1월호에서 공통적으로 강조하는 시장 리스크 및 관련 내용 원문 전체       │
│  Knowledge Retrieved:                                                                                           │
│  Additional Information: `                                                                                      │
│  KB주택시장리뷰                                                                                                 │
│  1                                                                                                              │
│  2026.                                                                                                          │
│  부동산연구팀                                                                                                   │
│  2015 9                                                                                                         │
│  2015 9                                                                                                         │
│  목 차                                                                                                          │
│  주택시장 동향                                                                                                  │
│  · 매매 가격 ………………………………………………… 4                                                                              │
│  · 매매 거래 …………………………………………………… 5                                                                             │
│  · 전세시장 …………………………………………………… 6                                                                              │
│  · 월세시장 …………………………………………………… 7                                                                              │
│  · 주택 분양 …………………………………………………… 8                                                                             │
│  · 주택 금융 …………………………………………………… 9                                                                             │
│  2                                                                                                              │
│  요 약                                                                                                          │
│  ◦ 2025년 수도권을 중심으로 주택 매매가격이 크게 상승했으나, 규제 강화에 따른 매수 심리 위축                    │
│  과 거래량 감소로 가격 상승폭은 빠르게 둔화                                                                     │
│  ∙ 10월 15일 정부의 규제 지역 확대 및 대출 규제 강화 이후 서울과 수도권을 중심으로 매수 심리가 위축             │
│  되면서 거래량이 빠르게 감소하고, 주요 아파트의 가격 상승폭이 크게 둔화되며 상승 기대감도 꺾임                  │
│  ∙ 전세가격은 연간 1.01% 상승하며 2024년 대비 상승폭이 ...                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  KB 주택시장 리뷰 문서를 기반으로 다음 질문에 답하라:                                                           │
│  2025년 12월호와 2026년 1월호에서 공통으로 강조하는 시장 리스크는 무엇인가?                                     │
│  Agent:                                                                                                         │
│  리서치 에이전트                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

2025년 12월호와 2026년 1월호에서 공통으로 강조하는 시장 리스크는 정부의 대출 규제 강화와 규제 지역 확대, 갭투자 금지 등으로 인해 서울과 수도권을 중심으로 매수 심리가 위축되고 주택 매매 거래량이 크게 감소하고 있다는 점이다. 이로 인해 주요 아파트 가격 상승폭이 빠르게 둔화되고 상승 기대감도 저하되고 있다. 또한 신규 입주 물량 감소와 전세 공급 부족 현상이 맞물리면서 전세가격 상승 압력이 지속되어 전세시장 불안이 당분간 이어질 가능성이 크며, 반전세 및 월세 수요 증가와 1·2인 가구 증가에 따른 월세 시장의 상승세가 지속되고 있는 점도 주요 리스크로 지적된다. 이와 함께 지역별 주택 분양시장의 양극화가 심화되고 미분양 물량이 일정 수준에서 횡보하는 현상도 시장 불안 요인으로 부각되고 있다.


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  f8f8a738-7f88-42c4-9b20-5f8634d96163                                                                           │
│  Final Output: 2025년 12월호와 2026년 1월호에서 공통으로 강조하는 시장 리스크는 정부의 대출 규제 강화와 규제    │
│  지역 확대, 갭투자 금지 등으로 인해 서울과 수도권을 중심으로 매수 심리가 위축되고 주택 매매 거래량이 크게       │
│  감소하고 있다는 점이다. 이로 인해 주요 아파트 가격 상승폭이 빠르게 둔화되고 상승 기대감도 저하되고 있다. 또한  │
│  신규 입주 물량 감소와 전세 공급 부족 현상이 맞물리면서 전세가격 상승 압력이 지속되어 전세시장 불안이 당분간    │
│  이어질 가능성이 크며, 반전세 및 월세 수요 증가와 1·2인 가구 증가에 따른 월세 시장의 상승세가 지속되고 있는     │
│  점도 주요 리스크로 지적된다. 이와 함께 지역별 주택 분양시장의 양극화가 심화되고 미분양 물량이 일정 수준에서    │
│  횡보하는 현상도 시장 불안 요인으로 부각되고 있다.                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



╭────────────────────────────── Execution Traces ──────────────────────────────╮
│                                                                              │
│  🔍 Detailed execution traces are available!                                 │
│                                                                              │
│  View insights including:                                                    │
│    • Agent decision-making process                                           │
│    • Task execution flow and timing                                          │
│    • Tool usage details                                                      │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯
Would you like to view your execution traces? [y/N] (20s timeout): 

╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                      

In [39]:
print(result.raw)


2025년 12월호와 2026년 1월호에서 공통으로 강조하는 시장 리스크는 정부의 대출 규제 강화와 규제 지역 확대, 갭투자 금지 등으로 인해 서울과 수도권을 중심으로 매수 심리가 위축되고 주택 매매 거래량이 크게 감소하고 있다는 점이다. 이로 인해 주요 아파트 가격 상승폭이 빠르게 둔화되고 상승 기대감도 저하되고 있다. 또한 신규 입주 물량 감소와 전세 공급 부족 현상이 맞물리면서 전세가격 상승 압력이 지속되어 전세시장 불안이 당분간 이어질 가능성이 크며, 반전세 및 월세 수요 증가와 1·2인 가구 증가에 따른 월세 시장의 상승세가 지속되고 있는 점도 주요 리스크로 지적된다. 이와 함께 지역별 주택 분양시장의 양극화가 심화되고 미분양 물량이 일정 수준에서 횡보하는 현상도 시장 불안 요인으로 부각되고 있다.


In [ ]:
# 계획

In [ ]:
from crewai import Agent
from crewai import Task
from crewai import Crew, Process

# Agent
analyst = Agent(
    role="데이터 분석가",
    goal="수집된 정보를 기반으로 핵심 인사이트를 도출합니다.",
    backstory="복잡한 데이터에서도 패턴과 의미를 찾아내는 능력이 탁월합니다."
)

writer = Agent(
    role="보고서 작성가",
    goal="전체 분석 내용을 하나의 구조화된 문서로 작성합니다.",
    backstory="명확하고 간결한 표현으로 정보를 전달하는 데 강점을 지닙니다."
)

# Task
analyze_task = Task(
    description=(
        "'{topic}'에 대해 수집된 정보를 분석하여 핵심 인사이트 3가지를 도출하세요."
    ),
    expected_output="핵심 인사이트 목록.",
    agent=analyst
)

write_task = Task(
    description=(
        "'{topic}'에 대한 분석 내용을 기반으로 간단한 보고서를 작성하세요."
    ),
    expected_output="300자 내외 보고서 문단.",
    agent=writer
)

# Crew
crew = Crew(
    agents=[analyst, writer],
    tasks=[analyze_task, write_task],
    process=Process.sequential,
    planning=True,
    verbose=True
)

result = crew.kickoff(
    inputs={"topic": "2026년 AI 트렌드"}
)
print(result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  7b89887f-5f5d-4aeb-a95d-1073a612822b                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


[2026-01-20 21:53:55][INFO]: Planning the crew execution


╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Based on these tasks summary:                                                                            │
│                  Task Number 1 - '2026년 AI 트렌드'에 대해 수집된 정보를 분석하여 핵심 인사이트 3가지를         │
│  도출하세요.                                                                                                    │
│                  "task_description": '2026년 AI 트렌드'에 대해 수집된 정보를 분석하여 핵심 인사이트 3가지를     │
│  도출하세요.                                                                                                    │
│                  "task_expected_output": 핵심 인사이트 목록.                                                    │
│                  "agent": 데이터 분석가                                                                         │
│                  "agent_goal": 수집된 정보를 기반으로 핵심 인사이트를 도출합니다.                               │
│                  "task_tools": []                                                                               │
│                  "agent_tools": "agent has no tools"                                                            │
│                  Task Number 2 - '2026년 AI 트렌드'에 대한 분석 내용을 기반으로 간단한 보고서를 작성하세요.     │
│                  "task_description": '2026년 AI 트렌드'에 대한 분석 내용을 기반으로 간단한 보고서를             │
│  작성하세요.                                                                                                    │
│                  "task_expected_output": 300자 내외 보고서 문단.                                                │
│                  "agent": 보고서 작성가                                                                         │
│                  "agent_goal": 전체 분석 내용을 하나의 구조화된 문서로 작성합니다.                              │
│                  "task_tools": []                                                                               │
│                  "agent_tools": "agent has no tools"                                                            │
│   Create the most descriptive plan based on the tasks descriptions, tools available, and agents' goals for      │
│  them to execute their goals with perfection.                                                                   │
│  ID: 4e897b01-c941-4c83-897c-98ba0a306565                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: '2026년 AI 트렌드'에 대해 수집된 정보를 분석하여 핵심 인사이트 3가지를 도출하세요.1. Begin by reviewing  │
│  all collected information concerning '2026년 AI 트렌드'. This should include articles, research papers, and    │
│  relevant data sources. 2. Identify key themes or patterns present within the collected data. Pay attention to  │
│  recurring topics or significant statistical data that reflect emerging trends. 3. Make notes of potential      │
│  insights by summarizing findings based on the analysis of these themes. 4. Organize these insights into a      │
│  structured format, clearly identifying at least three major insights with brief explanations for each. 5.      │
│  Ensure that insights are supported by the analyzed data, maintaining a logical connection between the trends   │
│  observed and the insights drawn. 6. Finalize the list of core insights by reviewing and editing for clarity    │
│  and conciseness, ensuring they are actionable and relevant to stakeholders.                                    │
│  ID: 57a9f8f8-946e-4297-ad22-9b03ac15f8ef                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Based on these tasks summary:                                                                                  │
│                  Task Number 1 - '2026년 AI 트렌드'에 대해 수집된 정보를 분석하여 핵심 인사이트 3가지를         │
│  도출하세요.                                                                                                    │
│                  "task_description": '2026년 AI 트렌드'에 대해 수집된 정보를 분석하여 핵심 인사이트 3가지를     │
│  도출하세요.                                                                                                    │
│                  "task_expected_output": 핵심 인사이트 목록.                                                    │
│                  "agent": 데이터 분석가                                                                         │
│                  "agent_goal": 수집된 정보를 기반으로 핵심 인사이트를 도출합니다.                               │
│                  "task_tools": []                                                                               │
│                  "agent_tools": "agent has no tools"                                                            │
│                  Task Number 2 - '2026년 AI 트렌드'에 대한 분석 내용을 기반으로 간단한 보고서를 작성하세요.     │
│                  "task_description": '2026년 AI 트렌드'에 대한 분석 내용을 기반으로 간단한 보고서를             │
│  작성하세요.                                                                                                    │
│                  "task_expected_output": 300자 내외 보고서 문단.                                                │
│                  "agent": 보고서 작성가                                                                         │
│                  "agent_goal": 전체 분석 내용을 하나의 구조화된 문서로 작성합니다.                              │
│                  "task_tools": []                                                                               │
│                  "agent_tools": "agent has no tools"                                                            │
│   Create the most descriptive plan based on the tasks descriptions, tools available, and agents' goals for      │
│  them to execute their goals with perfection.                                                                   │
│  Agent:                                                                                                         │
│  Task Execution Planner                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): 



╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                                                              │
│  Info: Tracing has been disabled.                                            │
│                                                                              │
│  Your preference has been saved. Future Crew/Flow executions will not        │
│  collect traces.                                                             │
│                                                                              │
│  To enable tracing later, do any one of these:                               │
│  • Set tracing=True in your Crew/Flow code                                   │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file               │
│  • Run: crewai traces enable                                                 │
│                                                                              │
╰─────────────────────────

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  7b89887f-5f5d-4aeb-a95d-1073a612822b                                                                           │
│  Final Output: 2026년 AI 트렌드는 지능형 맞춤형 서비스의 확대, 강화된 AI 윤리 및 규제 대응, 다중 모달 AI와      │
│  휴먼-기계 협업 가속화가 중심입니다. 개인 맞춤형 솔루션은 사용자 경험을 혁신하며, 엄격한 윤리 기준은 신뢰 있는  │
│  AI 생태계 조성을 돕습니다. 더불어 다중 모달 AI는 복합 데이터 처리를 통한 효율적 협업 환경을 만들어 생산성과    │
│  혁신을 촉진합니다. 이 세 가지 인사이트는 향후 AI 전략과 정책 수립에 핵심적 의의를 지닙니다.                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [ ]:
# 5.11.	CLI로 CrewAI Flow 프로젝트 생성하기

In [ ]:
# 터미널에서 실행 
"""
# 1) MAC/Linux 로컬 가상환경 활성화
source .venv/bin/activate	# Windows라면 .venv\Scripts\activate

# 1) Flow 프로젝트 스캐폴딩 생성
crewai create flow guide_creator_flow
cd guide_creator_flow

# 2) 의존성 설치 및 가상환경 생성
crewai install

crewai flow plot
"""

In [ ]:
# CrewAI 실전 프로젝트: 주식 분석 에이전트 시스템 구축하기 

In [52]:
import sys
!{sys.executable} -m pip install yfinance

In [53]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [54]:
from crewai_tools import ScrapeWebsiteTool, SerperDevTool
from crewai import Agent, Task, Crew

search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()

In [55]:
import yfinance as yf
from crewai.tools import tool

@tool("Stock News")
def stock_news(ticker: str) -> list:
    """주어진 티커의 최신 뉴스를 반환합니다."""
    return yf.Ticker(ticker).news

@tool("Stock Price")
def stock_price(ticker: str):
    """주어진 티커의 최근 1개월 주가 데이터를 반환합니다."""
    return yf.Ticker(ticker).history(period="1mo")

@tool("Income Statement")
def income_stmt(ticker: str):
    """기업의 손익계산서를 반환합니다."""
    return yf.Ticker(ticker).income_stmt

@tool("Balance Sheet")
def balance_sheet(ticker: str):
    """기업의 재무상태표를 반환합니다."""
    return yf.Ticker(ticker).balance_sheet

@tool("Insider Transactions")
def insider_transactions(ticker: str):
    """기업 내부자의 주식 거래 내역을 반환합니다."""
    return yf.Ticker(ticker).insider_transactions


In [56]:
researcher_agent = Agent(
    role="리서치 분석가",
    goal="기업의 최신 뉴스와 시장 심리를 수집하고 해석합니다.",
    backstory="각종 뉴스와 온라인 정보를 분석해 핵심 인사이트를 발굴하는 데이터 리서처입니다.",
    verbose=True,
    allow_delegation=True,
    tools=[stock_news, scrape_tool, search_tool]
)

technical_analyst_agent = Agent(
    role="기술적 분석가",
    goal="주가 흐름을 분석하고 추세, 지지선, 저항선을 식별합니다.",
    backstory="차트 해석과 패턴 분석에 능숙한 기술적 분석 전문가입니다.",
    verbose=True,
    allow_delegation=True,
    tools=[stock_price]
)

financial_analyst_agent = Agent(
    role="재무 분석가",
    goal="재무제표, 내부자 거래 정보 등을 분석해 기업의 재무 건전성을 평가합니다.",
    backstory="기업의 실적과 재무 구조를 기반으로 투자 판단을 돕는 전문가입니다.",
    verbose=True,
    allow_delegation=True,
    tools=[income_stmt, balance_sheet, insider_transactions]
)

hedge_fund_manager_agent = Agent(
    role="헤지펀드 매니저",
    goal="각 분석 결과를 종합해 최종 투자 전략을 결정합니다.",
    backstory="오랜 경험을 통해 시장을 읽고 포트폴리오를 최적화하는 투자 전문가입니다.",
    verbose=True,
    allow_delegation=True,
)

In [57]:
researcher_task = Task(
    description=(
        "기업 {company}와 관련된 최신 뉴스 및 시장 심리를 조사하고 분석한 뒤 "
        "핵심 내용을 요약하세요."
    ),
    expected_output="뉴스 요약과 시장 심리 변화에 대한 분석 결과.",
    agent=researcher_agent,
)


technical_analysis_task = Task(
    description=(
        "기업 {company}의 최근 주가 데이터를 분석하여 "
        "지지선, 저항선, 주요 추세, 잠재적 진입 포인트를 식별하세요."
    ),
    expected_output="기술적 분석 보고서.",
    agent=technical_analyst_agent,
)


financial_analysis_task = Task(
    description=(
        "기업 {company}의 재무제표(손익계산서·재무상태표)와 "
        "내부자 거래 데이터를 분석하여 재무 건전성과 실적 흐름을 평가하세요."
    ),
    expected_output="재무 분석 보고서.",
    agent=financial_analyst_agent,
)


investment_recommendation = Task(
    description=(
        "뉴스 분석, 기술적 분석, 재무 분석 결과를 모두 종합하여 "
        "{company} 기업에 대한 최종 투자 전략을 작성하세요. "
        "BUY, SELL, HOLD 중 하나를 선택하고, 해당 판단의 근거를 명확히 제시해야 합니다. "
        "지정된 가격대(지지·저항 수준)를 근거로 구체적인 매매 전략을 포함하고, "
        "마지막에는 내용을 요약하세요."
    ),
    expected_output=(
        "한국어로 작성합니다."
        "BUY/SELL/HOLD 중 하나를 선택한 투자 전략 보고서. "
        "근거, 목표 가격, 지지·저항선 등 필수 정보를 포함해야 합니다. "
        "보고서 말미에는 요약문을 포함합니다."
    ),
    agent=hedge_fund_manager_agent,
    context=[
        researcher_task,
        technical_analysis_task,
        financial_analysis_task,
    ],
    ouput_file="investment_recommendation.md"
)


In [ ]:
from crewai import Process

financial_trading_crew = Crew(
    agents=[
        researcher_agent,
        technical_analyst_agent,
        financial_analyst_agent,
        hedge_fund_manager_agent
    ],
    tasks=[
        researcher_task,
        technical_analysis_task,
        financial_analysis_task,
        investment_recommendation
    ],
    manager_llm="gpt-4o",
    process=Process.hierarchical,
    verbose=True
)

result = financial_trading_crew.kickoff(inputs={'company': 'NVDA'})


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  c0f99da9-f0be-42f9-991b-c548b879df6b                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 기업 NVDA와 관련된 최신 뉴스 및 시장 심리를 조사하고 분석한 뒤 핵심 내용을 요약하세요.                   │
│  ID: 9dd67217-c16b-47f3-873a-42d0f8f4398d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Task: 기업 NVDA와 관련된 최신 뉴스 및 시장 심리를 조사하고 분석한 뒤 핵심 내용을 요약하세요.                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: Stock News                                                                                               │
│  Args: {"ticker": "NVDA"}                                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  [{'id': 'ac1b7cd7-5c3a-41f6-84cc-9f27df508da0', 'content': {'id': 'ac1b7cd7-5c3a-41f6-84cc-9f27df508da0',      │
│  'contentType': 'STORY', 'title': "'A broadening playbook': Wall Street sees stock market gains beyond tech",   │
│  'description': '', 'summary': "Enthusiasm for artificial intelligence's prospects will continue to drive the   │
│  market higher in 2026. The gains won't be limited to tech stocks, strategists say.", 'pubDate':                │
│  '2026-01-18T16:05:56Z', 'displayTime': '2026-01-19T15:15:59Z', 'isHosted': True, 'bypassModal': False,         │
│  'previewUrl': None, 'thumbnail': {'originalUrl':                                                               │
│  'https://s.yimg.com/os/creatr-uploaded-images/2026-01/b7798ba0-f2f9-11f0-bdfd-6f91897d0d7a', 'originalWidth':  │
│  4898, 'originalHeight': 3265, 'caption': '', 'resolutions': [{'url':                                           │
│  'https://s.yimg.com/uu/api/res/1.2/uZ.qge_FqrXSfvbSd2LCcg--~B/aD0zMjY1O3c9NDg5ODthcHBpZD15dGFjaHlvbg--/https:  │
│  //s.yimg.com/os/creatr-uploaded-images/2026-01/b7798ba0-f2f9-11f0-bdfd-6f91897d0d7a', 'width': 4898,           │
│  'height': 3265, 'tag': 'original'}, {'url':                                                                    │
│  'https://s.yimg.com/uu/api/res/1.2/lrdx8vbEfVq4s7sy9ZluYg--~B/Zmk9c3RyaW07aD0xMjg7dz0xNzA7YXBwaWQ9eXRhY2h5b24  │
│  -/https://s.yimg.com/os/creatr-uploaded-images/2026-01/b7798ba0-f2f9-11f0-bdfd-6f91897d0d7a', 'width': 170,    │
│  'height': 128, 'tag': '170x128'}]}, 'provider': {'displayName': 'Yahoo Finance', 'url':                        │
│  'http://finance.yahoo.com/'}, 'canonicalUrl': {'url':                                                          │
│  'https://finance.yahoo.com/news/a-broadening-playbook-wall-street-sees-stock-market-gains-beyond-tech-1605564  │
│  66.html', 'site': 'finance', 'region': 'US', 'lang': 'en-US'}, 'clickThroughUrl': {'url':                      │
│  'https://finance.yahoo.com/news/a-broadening-playbook-wall-street-sees-stock-market-gains-beyond-tech-1605564  │
│  66.html', 'site': 'finance', 'region': 'US', 'lang': 'en-US'}, 'metadata': {'editorsPick': True}, 'finance':   │
│  {'premiumFinance': {'isPremiumNews': False, 'isPremiumFreeNews': False}}, 'storyline': None}}, {'id':          │
│  '23ce0232-2443-4d10-ba93-e2b96dbe4e93', 'content': {'id': '23ce0232-2443-4d10-ba93-e2b96dbe4e93',              │
│  'contentType': '...                                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **최종 투자 전략 보고서: NVIDIA (NVDA) 기업에 대한 투자 전략**                                                 │
│                                                                                                                 │
│  **1. 최근 뉴스 및 시장 심리:**                                                                                 │
│  NVIDIA는 AI 혁신에서 주도적인 역할을 하며, 다방면의 기회를 창출하여 긍정적인 시장 심리를 얻고 있습니다. AI     │
│  주도 성장 서사에 대한 투자자 신뢰가 높으나, 기술 섹터의 과대평가와 거품 발생 가능성으로 인한 변동성에 대한     │
│  주의도 필요합니다.                                                                                             │
│                                                                                                                 │
│  **2. 기술적 분석:**                                                                                            │
│  주가는 현재 $183–$190의 범위 내에서 조정 중입니다. $182.35와 $183.14의 주요 지지선은 견고하게 유지되고         │
│  있으며, 이는 저가 매수 기회를 형성합니다. 반대로 $189.21과 $190.44의 저항이 단기적으로 작용하고 있으며,        │
│  $190을 상향 돌파하는 경우 상승세가 예상됩니다.                                                                 │
│                                                                                                                 │
│  **3. 재무 분석:**                                                                                              │
│  NVIDIA는 매우 강력한 재무 성과를 보이고 있으며, 연 매출은 2023년 $27.0 billion에서 2025년 $130.5 billion으로   │
│  급증했습니다. 순이익 또한 $4.37 billion에서 $72.88 billion으로 증가했습니다. 회사의 유동성은 충분하며, 부채는  │
│  감소했습니다. 내부자가 $184–$192에서 주식을 매도한 것은 문맥상 정상적인 차익 실현으로 보입니다.                │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  **투자 전략 및 추천:**                                                                                         │
│                                                                                                                 │
│  - **포지셔닝:** 강력한 펀더멘털과 낙관적인 시장 심리, 그리고 전략적 기술적 지지를 고려할 때, $183~$184         │
│  범위에서 저가 매수에 진입하는 것이 적절합니다.                                                                 │
│                                                                                                                 │
│  - **진입 시점:** $183~$184까지의 하락을 매수 기회로 활용합니다. $190 이상에서의 신규 진입은 볼륨과 함께        │
│  명확한 돌파가 확인될 때까지 기다립니다.                                                                        │
│                                                                                                                 │
│  - **돌파 감시:** $190.50 이상에서의 지속적인 견지와 강한 볼륨이 나타날 경우 추가 매수 신호로 활용 가능합니다.  │
│                                                                                                                 │
│  - **위험 관리:** 보호 자산을 위해 손절매를 $181.50 아래에 설정하여 하방 조정 보호합니다. 기술 시장의           │
│  스트레스나 거품 징후를 면밀히 모니터링합니다.                                                                  │
│                                                                                                                 │
│  - **보유 전략:** 기존 주주들은 지지 구간에서 보유를 유지하며, $182 아래로 하락하는 경우 일부 처분 고려가       │
│  필요합니다.                                                                                                    │
│                                               

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  뉴스 분석, 기술적 분석, 재무 분석 결과를 모두 종합하여 NVDA 기업에 대한 최종 투자 전략을 작성하세요. BUY,      │
│  SELL, HOLD 중 하나를 선택하고, 해당 판단의 근거를 명확히 제시해야 합니다. 지정된 가격대(지지·저항 수준)를      │
│  근거로 구체적인 매매 전략을 포함하고, 마지막에는 내용을 요약하세요.                                            │
│  Agent:                                                                                                         │
│  Crew Manager                                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  c0f99da9-f0be-42f9-991b-c548b879df6b                                                                           │
│  Final Output: **최종 투자 전략 보고서: NVIDIA (NVDA) 기업에 대한 투자 전략**                                   │
│                                                                                                                 │
│  **1. 최근 뉴스 및 시장 심리:**                                                                                 │
│  NVIDIA는 AI 혁신에서 주도적인 역할을 하며, 다방면의 기회를 창출하여 긍정적인 시장 심리를 얻고 있습니다. AI     │
│  주도 성장 서사에 대한 투자자 신뢰가 높으나, 기술 섹터의 과대평가와 거품 발생 가능성으로 인한 변동성에 대한     │
│  주의도 필요합니다.                                                                                             │
│                                                                                                                 │
│  **2. 기술적 분석:**                                                                                            │
│  주가는 현재 $183–$190의 범위 내에서 조정 중입니다. $182.35와 $183.14의 주요 지지선은 견고하게 유지되고         │
│  있으며, 이는 저가 매수 기회를 형성합니다. 반대로 $189.21과 $190.44의 저항이 단기적으로 작용하고 있으며,        │
│  $190을 상향 돌파하는 경우 상승세가 예상됩니다.                                                                 │
│                                                                                                                 │
│  **3. 재무 분석:**                                                                                              │
│  NVIDIA는 매우 강력한 재무 성과를 보이고 있으며, 연 매출은 2023년 $27.0 billion에서 2025년 $130.5 billion으로   │
│  급증했습니다. 순이익 또한 $4.37 billion에서 $72.88 billion으로 증가했습니다. 회사의 유동성은 충분하며, 부채는  │
│  감소했습니다. 내부자가 $184–$192에서 주식을 매도한 것은 문맥상 정상적인 차익 실현으로 보입니다.                │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  **투자 전략 및 추천:**                                                                                         │
│                                                                                                                 │
│  - **포지셔닝:** 강력한 펀더멘털과 낙관적인 시장 심리, 그리고 전략적 기술적 지지를 고려할 때, $183~$184         │
│  범위에서 저가 매수에 진입하는 것이 적절합니다.                                                                 │
│                                                                                                                 │
│  - **진입 시점:** $183~$184까지의 하락을 매수 기회로 활용합니다. $190 이상에서의 신규 진입은 볼륨과 함께        │
│  명확한 돌파가 확인될 때까지 기다립니다.                                                                        │
│                                                                                                                 │
│  - **돌파 감시:** $190.50 이상에서의 지속적인 견지와 강한 볼륨이 나타날 경우 추가 매수 신호로 활용 가능합니다.  │
│                                                                                                                 │
│  - **위험 관리:** 보호 자산을 위해 손절매를 $181.50 아래에 설정하여 하방 조정 보호합니다. 기술 시장의           │
│  스트레스나 거품 징후를 면밀히 모니터링합니다.                                                                  │
│                                                                                                                 │
│  

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



╭────────────────────────────── Execution Traces ──────────────────────────────╮
│                                                                              │
│  🔍 Detailed execution traces are available!                                 │
│                                                                              │
│  View insights including:                                                    │
│    • Agent decision-making process                                           │
│    • Task execution flow and timing                                          │
│    • Tool usage details                                                      │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯
Would you like to view your execution traces? [y/N] (20s timeout): 

╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                      

In [60]:
print(result)

**최종 투자 전략 보고서: NVIDIA (NVDA) 기업에 대한 투자 전략**

**1. 최근 뉴스 및 시장 심리:**  
NVIDIA는 AI 혁신에서 주도적인 역할을 하며, 다방면의 기회를 창출하여 긍정적인 시장 심리를 얻고 있습니다. AI 주도 성장 서사에 대한 투자자 신뢰가 높으나, 기술 섹터의 과대평가와 거품 발생 가능성으로 인한 변동성에 대한 주의도 필요합니다.

**2. 기술적 분석:**  
주가는 현재 $183–$190의 범위 내에서 조정 중입니다. $182.35와 $183.14의 주요 지지선은 견고하게 유지되고 있으며, 이는 저가 매수 기회를 형성합니다. 반대로 $189.21과 $190.44의 저항이 단기적으로 작용하고 있으며, $190을 상향 돌파하는 경우 상승세가 예상됩니다. 

**3. 재무 분석:**  
NVIDIA는 매우 강력한 재무 성과를 보이고 있으며, 연 매출은 2023년 $27.0 billion에서 2025년 $130.5 billion으로 급증했습니다. 순이익 또한 $4.37 billion에서 $72.88 billion으로 증가했습니다. 회사의 유동성은 충분하며, 부채는 감소했습니다. 내부자가 $184–$192에서 주식을 매도한 것은 문맥상 정상적인 차익 실현으로 보입니다.

---

**투자 전략 및 추천:**

- **포지셔닝:** 강력한 펀더멘털과 낙관적인 시장 심리, 그리고 전략적 기술적 지지를 고려할 때, $183~$184 범위에서 저가 매수에 진입하는 것이 적절합니다.

- **진입 시점:** $183~$184까지의 하락을 매수 기회로 활용합니다. $190 이상에서의 신규 진입은 볼륨과 함께 명확한 돌파가 확인될 때까지 기다립니다.
  
- **돌파 감시:** $190.50 이상에서의 지속적인 견지와 강한 볼륨이 나타날 경우 추가 매수 신호로 활용 가능합니다.
  
- **위험 관리:** 보호 자산을 위해 손절매를 $181.50 아래에 설정하여 하방 조정 보호합니다. 기술 시장의 스트레스나 거품 징후를 면밀히 모니터링

In [61]:
import os
from crewai import Crew

# Set custom storage location
os.environ["CREWAI_STORAGE_DIR"] = "./storage"

In [62]:
from crewai import Agent, Crew, Task, Process

assistant = Agent(
    role="대화형 어시스턴트",
    goal="사용자의 정보를 기억하고 다음 대화에서 활용한다.",
    backstory="사용자의 선호를 잘 기억하는 비서이다."
)

task = Task(
    description="사용자의 질문에 답변하라: {question}",
    expected_output="간단한 답변.",
    agent=assistant,
)

crew = Crew(
    agents=[assistant],
    tasks=[task],
    process=Process.sequential,
    memory=True
)

# 첫 대화: 기억 저장됨
crew.kickoff(inputs={"question": "내 이름은 김철수야. 기억해줘"})

# 두 번째 대화: 기억을 불러와야 함
result = crew.kickoff(inputs={"question": "내 이름이 뭐였지?"})

print(result.raw)

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): 사용자의 이름은 김철수입니다.


╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): 



╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                                                              │
│  Info: Tracing has been disabled.                                            │
│                                                                              │
│  Your preference has been saved. Future Crew/Flow executions will not        │
│  collect traces.                                                             │
│                                                                              │
│  To enable tracing later, do any one of these:                               │
│  • Set tracing=True in your Crew/Flow code                                   │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file               │
│  • Run: crewai traces enable                                                 │
│                                                                              │
╰─────────────────────────

In [63]:
import os
import chromadb
from crewai.utilities.paths import db_storage_path

storage_path = db_storage_path()
chroma_path = os.path.join(storage_path, "long_term_memory")

client = chromadb.PersistentClient(path=chroma_path)
print(client.list_collections())


[]


In [64]:
import os
from crewai.utilities.paths import db_storage_path

storage_path = db_storage_path()
print(f"Storage path: {storage_path}")
print(f"Path exists: {os.path.exists(storage_path)}")
print(f"Is writable: {os.access(storage_path, os.W_OK) if os.path.exists(storage_path) else 'Path does not exist'}")

# Create with proper permissions
if not os.path.exists(storage_path):
    os.makedirs(storage_path, mode=0o755, exist_ok=True)
    print(f"Created storage directory: {storage_path}")

Storage path: /Users/a453866/Library/Application Support/storage
Path exists: True
Is writable: True


In [65]:
import chromadb
from crewai.utilities.paths import db_storage_path

# Connect to CrewAI's ChromaDB
storage_path = db_storage_path()
chroma_path = os.path.join(storage_path)

if os.path.exists(chroma_path):
    client = chromadb.PersistentClient(path=chroma_path)
    collections = client.list_collections()

    print("ChromaDB Collections:")
    for collection in collections:
        print(f"  - {collection.name}: {collection.count()} documents")
else:
    print("No ChromaDB storage found")

ChromaDB Collections:
